Notebook to generate data.h5 that includes curve_data, sample_info and igi_gene_call datasets

In [11]:
import os
import json
import glob
import pandas as pd

In [26]:
DATA_DIR = '/Users/huongvu/Desktop/pcr/Data/New Data/'
CURVE_DIR = os.path.join(DATA_DIR, 'extracted_csv')
PRIMARY_DIR = os.path.join(DATA_DIR, 'primary_samples')
SIM_DIR = '/Users/huongvu/Desktop/PCR_simulation/'


In [14]:
sample_info = pd.read_csv(os.path.join(PRIMARY_DIR, 'sample_info.csv'))
sample_gene = pd.read_csv(os.path.join(PRIMARY_DIR, 'sample_gene.csv'))

In [22]:
folders = os.listdir(CURVE_DIR)
folders.remove('.DS_Store')

amp_df = pd.DataFrame()
multcomp_df = pd.DataFrame()
da_result_df = pd.DataFrame()

error_files = {'amplification':[],
               'multicomponent':[],
               'da_result':[]}

In [23]:
for folder in folders:
    try: 
        df = pd.read_csv(os.path.join(CURVE_DIR, folder, 'Amplification Data.csv'))
        df = df.loc[:, ~df.columns.isin(['Well','Sample','Omit'])]
        df.columns = ['well_position','cycle_no','target','rn','drn']
        df.loc[:,'file'] = folder
        amp_df = pd.concat([amp_df, df],
                            ignore_index=True)
    except Exception as e:
        error_files['amplification'].append({'folder':folder,
                                          'msg': str(e)}) 
    
    try:
        df = pd.read_csv(os.path.join(CURVE_DIR, folder, 'Multicomponent.csv'))
        df = df.drop(columns='Well')
        df = (df.set_index(['Well Position','Cycle Number'])
                .melt(ignore_index=False)
                .reset_index())
        df.columns = ['well_position','cycle_no','dye','Fn']
        df.loc[:,'file'] = folder
        multcomp_df = pd.concat([multcomp_df, df],
                                ignore_index=True)
    except Exception as e:
        error_files['multicomponent'].append({'folder':folder,
                                          'msg': str(e)})

    try:
        df = pd.read_csv(os.path.join(CURVE_DIR, folder, 'Results.csv'))
        df = df[['Well Position','Target','Reporter','Amp Score','Cq',
                 'Threshold','Baseline Start','Baseline End']]
        df.columns = ['well_position','target','dye','amp_score','cq',
                      'threshold','baseline_start','baseline_end']
        df.loc[:,'file'] = folder
        da_result_df = pd.concat([da_result_df, df],
                                 ignore_index=True)
    except Exception as e:
        error_files['da_result'].append({'folder':folder,
                                          'msg': str(e)})

In [24]:
# check for error files
print(error_files)

{'amplification': [], 'multicomponent': [], 'da_result': []}


In [25]:
join_df = (da_result_df
           .merge(amp_df, how = 'inner', on = ['well_position','file','target'])
           .merge(multcomp_df, how = 'inner', on = ['well_position','file','dye','cycle_no']))

file_dict = sample_info[['file','pcr_plate']].drop_duplicates().set_index('file').to_dict()['pcr_plate']
join_df['pcr_plate'] = join_df['file'].replace(file_dict)

join_df['curve_idx'] = join_df.groupby(['pcr_plate','target','well_position']).ngroup()
join_df = join_df.drop('file', axis = 1)

sample_info = sample_info.drop(sample_info[sample_info.pcr_plate == 'AC00DB15'].index, axis=0)
join_df = join_df.drop(join_df[join_df.pcr_plate == 'AC00DB15'].index, axis = 0)

In [27]:
join_df.to_hdf(os.path.join(SIM_DIR, 'data', 'data.h5'), key = 'curve_data', mode='w')
sample_info.to_hdf(os.path.join(SIM_DIR, 'data', 'data.h5'), key='sample_info')
sample_gene.to_hdf(os.path.join(SIM_DIR, 'data', 'data.h5'), key='igi_gene_call')

/opt/anaconda3/envs/pcr/lib/python3.8/site-packages/pandas/core/generic.py:2703: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['sample_id', 'sample_barcode', 'pcr_plate', 'well_position',
       'sample_type', 'final_patient_result', 'current_sample_result',
       'created_date', 'record_type', 'retest_sample_id_1',
       'retest_sample_id_2', 'file'],
      dtype='object')]

  pytables.to_hdf(
